# RSNA Knee — submission notebook (probe #2: exp005 weights)

**This is a measuring instrument, not a final product** (CLAUDE.md §5/§9 decision, 2026-09-08). Originally probe #1 used exp002 (DINOv2-S on `labels_v2_final`); **updated 2026-09-09 to exp005** (same DINOv2-S architecture, retrained on `labels_v3_candidate` — the image+report noisy-student blend, ACCEPTED after beating exp002 on both OOF-vs-original-labels and gold-58, see EXPERIMENTS.md). Same defensive/inference stack, only the weights changed. Do not iterate on this notebook beyond keeping it robust.

Expects two attached Kaggle Datasets:
- `rsna-knee-code` — the `rsna_knee` package (`dataset.py`, `model.py`, `preprocess.py`, `infer_core.py`)
- `rsna-knee-exp005-weights` — `fold0/best.pt` … `fold4/best.pt` (exp005, DINOv2-S on labels_v3_candidate)

Internet is OFF (code competition, CLAUDE.md §9) — no `pip install` from PyPI; DICOM decode relies on whatever `pydicom`/`pylibjpeg`/`gdcm` ships in the Kaggle Python image. If a wheel is missing, attach a wheels dataset and adjust the install cell below rather than assume internet access.

In [ ]:
import importlib.util
import os, sys, time, traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Diagnostic (2026-09-08): Kaggle's newer input-mount layout nests under
# /kaggle/input/{datasets,competitions}/... instead of the old flat /kaggle/input/<slug>/.
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
    if depth <= 3:
        print(root)
    if depth >= 3:
        dirs[:] = []

def _find_dir(root: str, name_contains: str) -> Path | None:
    for p in Path(root).rglob("*"):
        if p.is_dir() and name_contains in p.name:
            return p
    return None

CODE_DIR = _find_dir("/kaggle/input", "rsna-knee-code")
WEIGHTS_DIR = _find_dir("/kaggle/input", "rsna-knee-exp005-weights")
TEST_DIR = _find_dir("/kaggle/input", "rsna-knee-abnormality-detection")
OUT_PATH = Path("/kaggle/working/submission.csv")
print("Resolved CODE_DIR:", CODE_DIR, "contents:", list(CODE_DIR.iterdir()) if CODE_DIR else None)
print("Resolved WEIGHTS_DIR:", WEIGHTS_DIR, WEIGHTS_DIR.exists() if WEIGHTS_DIR else None)
print("Resolved TEST_DIR:", TEST_DIR, TEST_DIR.exists() if TEST_DIR else None)

assert CODE_DIR is not None, "Attach the rsna-knee-code dataset (searched under /kaggle/input)"
assert WEIGHTS_DIR is not None and WEIGHTS_DIR.exists(), "Attach the rsna-knee-exp005-weights dataset"
assert TEST_DIR is not None and TEST_DIR.exists(), "Attach the competition data"

# Register whatever directory actually holds __init__.py as the "rsna_knee" package regardless of
# its on-disk folder name or nesting depth — the dataset zip's extraction layout on Kaggle
# (flat vs. wrapped in an rsna_knee/ subfolder) turned out not to match the old flat
# /kaggle/input/<slug>/rsna_knee/ assumption, so stop assuming and just find __init__.py.
init_files = list(CODE_DIR.rglob("__init__.py"))
assert init_files, f"No __init__.py found anywhere under {CODE_DIR}"
pkg_dir = init_files[0].parent
print("Package dir:", pkg_dir, "contents:", list(pkg_dir.iterdir()))
spec = importlib.util.spec_from_file_location(
    "rsna_knee", pkg_dir / "__init__.py", submodule_search_locations=[str(pkg_dir)]
)
rsna_knee = importlib.util.module_from_spec(spec)
sys.modules["rsna_knee"] = rsna_knee
spec.loader.exec_module(rsna_knee)

from rsna_knee.dataset import TARGET_COLUMNS, _patch_size_for_backbone
from rsna_knee.infer_core import (
    WallClockGuard, decode_study_slots, nearest_patch_multiple, prior_row, run_study_inference,
)
from rsna_knee.model import SlotModel

print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())

In [ ]:
# Config — must match configs/exp005_dinov2s_slot_v3labels.yaml exactly (docs/EXPERIMENTS.md 2026-09-09 entry).
# Same architecture/backbone as exp002 — only the training labels changed (labels_v3_candidate).
BACKBONE = "vit_small_patch14_dinov2.lvd142m"
SLICES_PER_SERIES = 24
IMAGE_SIZE = 384
OUT_SIZE = nearest_patch_multiple(IMAGE_SIZE, _patch_size_for_backbone(BACKBONE))  # 378 for patch14

# 9h GPU budget (CLAUDE.md §1); leave generous margin for Kaggle scheduling variance and the
# submission-file write. ~1300 test studies — budgeting well under the hard cap.
BUDGET_SECONDS = 3 * 3600

# GPU-compatibility check (2026-09-08 finding): Kaggle can hand out an older GPU (observed: Tesla
# P100, compute capability sm_60) that the installed PyTorch build doesn't ship kernels for
# (this build's lowest supported arch is sm_70) — `torch.cuda.is_available()` still returns True,
# so naively using "cuda" here would run every forward pass into a CUDA error and silently fall
# back to label priors for EVERY study, producing a submission that measures the prior baseline
# instead of the model. CUDA/PTX is forward-compatible (a binary built for sm_70 can run on
# sm_75+) but not backward, so the correct check is: GPU capability >= the build's LOWEST
# supported arch.
DEVICE = "cpu"
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    cap = major * 10 + minor
    supported_caps = [int(a[3:]) for a in torch.cuda.get_arch_list() if a.startswith("sm_")]
    if supported_caps and cap >= min(supported_caps):
        DEVICE = "cuda"
    else:
        print(f"  [WARN] GPU {torch.cuda.get_device_name(0)} (sm_{cap}) is older than this torch "
              f"build's minimum supported sm_{min(supported_caps) if supported_caps else '?'} — "
              f"falling back to CPU.")
print("device:", DEVICE, "T:", SLICES_PER_SERIES, "out_size:", OUT_SIZE)

In [ ]:
def load_models(weights_dir: Path, backbone: str, device: str) -> list:
    models = []
    for fold_dir in sorted(weights_dir.glob("fold*")):
        ckpt_path = fold_dir / "best.pt"
        if not ckpt_path.exists():
            print(f"  [WARN] no best.pt in {fold_dir.name}, skipping this fold")
            continue
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model = SlotModel(backbone, num_targets=len(TARGET_COLUMNS), pretrained=False).to(device)
        model.load_state_dict(ckpt["ema"])
        model.eval()
        models.append(model)
        print(f"Loaded {fold_dir.name}: val_macro_auc={ckpt.get('val_macro_auc'):.4f} "
              f"gold58_macro_auc={ckpt.get('gold58_macro_auc'):.4f}")
    if not models:
        raise RuntimeError(f"No fold checkpoints loaded from {weights_dir} — cannot run inference.")
    return models


models = load_models(WEIGHTS_DIR, BACKBONE, DEVICE)
print(f"Ensemble: {len(models)}/5 folds loaded.")
# Defensive degrade (CLAUDE.md §9): fewer than 5 folds still produces a valid, just slightly
# weaker, ensemble — continuing rather than crashing the whole submission over one bad file.

In [ ]:
test_df = pd.read_csv(TEST_DIR / "test.csv")
series_df = pd.read_csv(TEST_DIR / "test_series.csv").rename(
    columns={"StudyInstanceUID": "study_uid", "SeriesInstanceUID": "series_uid"}
)
series_root = TEST_DIR / "test_series"
print(f"{len(test_df)} test studies, {len(series_df)} series.")

In [ ]:
guard = WallClockGuard(BUDGET_SECONDS)
rows = []
n_prior_fallback = n_dual_flip = n_partial_ensemble = 0

for i, study_uid in enumerate(test_df["StudyInstanceUID"]):
    row = {"StudyInstanceUID": study_uid}
    try:
        if guard.over(margin=0.1):
            # Degrade gracefully (CLAUDE.md §9): stop doing real inference once we're within 10%
            # of budget, fall back to priors for whatever studies remain rather than risk a
            # Kaggle timeout with zero submission.
            print(f"  [BUDGET] {guard.elapsed():.0f}s used of {BUDGET_SECONDS}s at study "
                  f"{i}/{len(test_df)} — priors for the rest.")
            raise TimeoutError("wall-clock budget exhausted")

        study_series = series_df[series_df["study_uid"] == study_uid].to_dict("records")
        slots, tag_sides = decode_study_slots(
            study_uid, study_series, series_root / study_uid, SLICES_PER_SERIES, OUT_SIZE
        )
        if not slots:
            raise RuntimeError("no series decoded successfully for this study")
        if all(s not in ("R", "L") for s in tag_sides):
            n_dual_flip += 1

        probs = run_study_inference(slots, tag_sides, models, DEVICE, SLICES_PER_SERIES, OUT_SIZE)
        row.update({c: float(p) for c, p in zip(TARGET_COLUMNS, probs)})
    except Exception as e:
        print(f"  [WARN] {study_uid}: {type(e).__name__}: {e} — using label priors")
        traceback.print_exc()
        n_prior_fallback += 1
        row.update({c: float(p) for c, p in zip(TARGET_COLUMNS, prior_row())})
    rows.append(row)
    if (i + 1) % 100 == 0 or (i + 1) == len(test_df):
        print(f"  {i + 1}/{len(test_df)} studies, {guard.elapsed():.0f}s elapsed "
              f"({guard.fraction_used():.1%} of budget)")

print(f"Done: {len(rows)} studies, {n_prior_fallback} full-prior fallbacks, "
      f"{n_dual_flip} dual-flip (unresolved side), {guard.elapsed():.0f}s total.")

In [ ]:
sub_df = pd.DataFrame(rows)[["StudyInstanceUID"] + TARGET_COLUMNS]

# Sanity checks — never let a malformed file reach scoring silently (CLAUDE.md §9: "a crash =
# no score", and a malformed/incomplete file is just as costly.
assert len(sub_df) == len(test_df), f"row count mismatch: {len(sub_df)} vs {len(test_df)} test studies"
assert set(sub_df["StudyInstanceUID"]) == set(test_df["StudyInstanceUID"]), "study UID mismatch"
assert not sub_df[TARGET_COLUMNS].isna().any().any(), "NaNs in submission"
assert (sub_df[TARGET_COLUMNS] >= 0).all().all() and (sub_df[TARGET_COLUMNS] <= 1).all().all(), \
    "probabilities out of [0,1]"
expected_header = ["StudyInstanceUID"] + TARGET_COLUMNS
assert list(sub_df.columns) == expected_header, f"header mismatch: {list(sub_df.columns)}"

sub_df.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}: {len(sub_df)} rows, all checks passed.")
sub_df.head()